# Experiment 5 — Cross-model comparative aggregation (§8)

No GPU required. Reads per-model bundles from Drive, aggregates them, and renders the 5 paper figures + summary table.


In [ ]:
# !pip install -q git+https://github.com/YOURUSER/mech_spoof.git
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from mech_spoof.experiments.exp5_comparative import aggregate_results
from mech_spoof.viz import (
    make_summary_table,
    plot_attack_prediction_scatter,
    plot_authority_refusal_cosine,
    plot_conflict_compliance_bars,
    plot_probe_accuracy_overlay,
    plot_token_trace,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/mech_spoof_results')
OUT = Path('/content/drive/MyDrive/mech_spoof_results/_aggregate')
OUT.mkdir(parents=True, exist_ok=True)

report = aggregate_results(results_root=DRIVE_ROOT, out_dir=OUT)
per_model = report.per_model
print(report.summary_table.to_string())

In [ ]:
plot_probe_accuracy_overlay(per_model, OUT / 'fig1_probe_accuracy_overlay.png')
plot_authority_refusal_cosine(per_model, OUT / 'fig2_authority_refusal_cosine.png')
plot_conflict_compliance_bars(per_model, OUT / 'fig3_conflict_compliance_bars.png')
plot_attack_prediction_scatter(per_model, OUT / 'fig4_attack_prediction_scatter.png')
make_summary_table(per_model, OUT / 'table1_summary.csv')

for model_key, data in per_model.items():
    exp4 = data.get('exp4_attacks')
    if not exp4:
        continue
    for i, t in enumerate(exp4.get('traces', []) or []):
        plot_token_trace(
            t['trace'], OUT / f'fig5_trace_{model_key}_{i}.png',
            title=f"{model_key} — {t.get('harmful_goal', '')[:60]}",
        )
print('wrote figures to', OUT)